# 01 — Exploratory analysis

This notebook demonstrates the `interpretant` API with synthetic data.
Run `make install` before executing.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = 'notebook'

from interpretant.drift.metrics import cosine_distance, neighborhood_shift
from interpretant.visualization.trajectory import build_trajectory_figure
from interpretant.visualization.neighbors import build_neighbors_table

In [ ]:
# ── Synthetic data ────────────────────────────────────────────────────────
VOCAB = ['network', 'atom', 'information', 'model', 'evolution', 'structure']
DECADES = [1960, 1970, 1980, 1990, 2000, 2010]
VECTOR_DIM = 300
SEED = 42

rng = np.random.default_rng(SEED)
base_vectors = {word: rng.standard_normal(VECTOR_DIM) for word in VOCAB}
base_vectors = {w: v / np.linalg.norm(v) for w, v in base_vectors.items()}

model: dict[int, dict[str, np.ndarray]] = {}
for i, decade in enumerate(DECADES):
    noise_scale = 0.05 * (i + 1)
    decade_vectors = {}
    for word in VOCAB:
        noise = rng.standard_normal(VECTOR_DIM) * noise_scale
        v = base_vectors[word] + noise
        decade_vectors[word] = v / np.linalg.norm(v)
    model[decade] = decade_vectors

print(f'Built synthetic model: {len(DECADES)} decades × {len(VOCAB)} words × {VECTOR_DIM} dims')

In [ ]:
# ── Drift table ───────────────────────────────────────────────────────────
rows = []
sorted_decades = sorted(model.keys())
for word in VOCAB:
    for i in range(1, len(sorted_decades)):
        t0 = sorted_decades[i - 1]
        t1 = sorted_decades[i]
        rows.append({
            'word': word,
            'decade': t1,
            'drift': cosine_distance(model[t0][word], model[t1][word]),
        })

drift_df = pd.DataFrame(rows)
drift_df.pivot(index='decade', columns='word', values='drift').round(4)

In [ ]:
# ── Trajectory plot ───────────────────────────────────────────────────────
word = 'network'
word_vectors = {decade: model[decade][word] for decade in sorted_decades}
fig = build_trajectory_figure(word, word_vectors)
fig.show()

In [ ]:
# ── Nearest neighbours ────────────────────────────────────────────────────
vectors_by_decade = {
    decade: model[decade] for decade in sorted_decades
}
build_neighbors_table(word, vectors_by_decade, k=5)